# DRGS: Depth-Regularized 3D Gaussian Splatting (CVPRW 2024)

**Paper**: [Depth-Regularized Optimization for 3D Gaussian Splatting in Few-Shot Images](https://arxiv.org/abs/2311.13398)

Blender 합성 데이터(Mitsubishi 로고)에 대해 Baseline 3DGS vs DRGS를 비교 실험합니다.

**이 데이터의 핵심 특성:**
- 단색 무채색 오브젝트 → photometric loss만으로는 geometry 구속이 어려움
- GT depth가 완벽 (Blender 렌더링) → depth regularization 효과를 깨끗하게 측정 가능
- 641개 뷰, 360도 커버리지 (azimuth -180~+180, elevation 0~60)
- 배경 제거 적용 (depth mask → 검정) → bg_color=[0,0,0]과 일치

---
## Section 0: Configuration

In [ ]:
import os

# ============================================================
# 실험 설정
# ============================================================

# 이 레포를 Colab에서 pull해서 사용
PROJECT_ROOT = "/content/DepthRegularizedGS"
REPO_URL = "https://github.com/BAEJUNHAK/DepthRegularizedGS.git"

# Google Drive
DRIVE_MOUNT = "/content/drive"
DRIVE_SAVE_DIR = os.path.join(DRIVE_MOUNT, "MyDrive", "DRGS_results")

# 데이터 Google Drive file IDs
RGB_GDRIVE_ID = "1dnj1s-mqIuS6OcdSr5CczBr9u8yBzUUB"
DEPTH_GDRIVE_ID = "1KmXCzBYv_mkPmZnWka1ThCZSNHTyivKQ"

# 데이터셋
SCENE_NAME = "mitsubishi"
DATA_DIR = os.path.join(PROJECT_ROOT, "data", SCENE_NAME)

# Train/Test split
TRAIN_RATIO = 0.8
SPLIT_SEED = 42

# 학습 하이퍼파라미터
KSHOT = 30
SEED = 3
RESOLUTION = 1
ITERATIONS = 30000

TEST_ITERS = "7000 15000 30000"
SAVE_ITERS = "7000 15000 30000"

# 출력 경로
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "output")
BASELINE_DIR = os.path.join(OUTPUT_DIR, f"{SCENE_NAME}_baseline_k{KSHOT}_s{SEED}")
DRGS_DIR = os.path.join(OUTPUT_DIR, f"{SCENE_NAME}_drgs_k{KSHOT}_s{SEED}")

print(f"Scene: {SCENE_NAME}")
print(f"K-shot: {KSHOT}, Seed: {SEED}")
print(f"Baseline: {BASELINE_DIR}")
print(f"DRGS:     {DRGS_DIR}")

---
## Section 1: Environment Setup

### 1.1 GPU 확인

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU 없음. Runtime > Change runtime type > GPU 선택")

print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB)")
print(f"CUDA: {torch.version.cuda}, PyTorch: {torch.__version__}")

### 1.2 레포 Pull & CUDA 빌드 (~3분)

커스텀 리더(`scene/custom_reader.py`)와 코드 패치가 이미 레포에 포함되어 있습니다.

In [ ]:
import os

if not os.path.exists(PROJECT_ROOT):
    !git clone {REPO_URL} --recursive {PROJECT_ROOT}
else:
    print(f"이미 존재: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)

# CUDA 서브모듈 빌드
!pip install -e submodules/diff-gaussian-rasterization-depth-acc
!pip install -e submodules/simple-knn

# 추가 의존성
!pip install -q plyfile==0.8.1 imageio lpips opencv-python-headless

### 1.3 설치 검증

In [ ]:
import torch
from diff_gaussian_rasterization_depth_acc import GaussianRasterizer
from simple_knn._C import distCUDA2
import plyfile, cv2, imageio

print(f"[OK] torch {torch.__version__}, CUDA {torch.version.cuda}")
print(f"[OK] rasterizer, simple-knn, plyfile, cv2 {cv2.__version__}")
print("\n모든 의존성 OK")

---
## Section 2: Data Preparation

Google Drive에서 RGB + Depth 다운로드 후 `data/mitsubishi/`에 배치합니다.

**커스텀 리더(`scene/custom_reader.py`)가 자동으로 처리하는 것:**
- `calib_*.ini` 파싱 → R, T, K
- `depth_raw_*.png` (uint16) → float depth (meters)
- 배경 제거: depth==0 픽셀 → 검정(0,0,0) (bg_color 일치, 배경 88.8% loss 낭비 방지)
- GT depth back-projection → 초기 point cloud
- ZoeDepth 불필요

### 2.1 Google Drive 마운트 & 데이터 다운로드

In [ ]:
from google.colab import drive
drive.mount(DRIVE_MOUNT)
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

In [ ]:
os.chdir(PROJECT_ROOT)
os.makedirs(DATA_DIR, exist_ok=True)

!pip install -q gdown

# RGB + Camera
!gdown "{RGB_GDRIVE_ID}" -O /tmp/dataset.zip
!unzip -q -o /tmp/dataset.zip -d /tmp/rgb_extract

# Depth
!gdown "{DEPTH_GDRIVE_ID}" -O /tmp/depth_data.zip
!unzip -q -o /tmp/depth_data.zip -d /tmp/depth_extract

# DATA_DIR로 합침
!cp /tmp/rgb_extract/dataset/rgb_*.png {DATA_DIR}/
!cp /tmp/rgb_extract/dataset/calib_*.ini {DATA_DIR}/
!cp /tmp/depth_extract/dataset_depth/depth_raw_*.png {DATA_DIR}/

!rm -rf /tmp/dataset.zip /tmp/depth_data.zip /tmp/rgb_extract /tmp/depth_extract

import glob
n_rgb = len(glob.glob(os.path.join(DATA_DIR, "rgb_*.png")))
n_depth = len(glob.glob(os.path.join(DATA_DIR, "depth_raw_*.png")))
n_calib = len(glob.glob(os.path.join(DATA_DIR, "calib_*.ini")))
print(f"RGB: {n_rgb}, Depth: {n_depth}, Calib: {n_calib}")

### 2.2 Train/Test Split 생성

In [ ]:
import json
import numpy as np

n_views = len(glob.glob(os.path.join(DATA_DIR, "calib_*.ini")))
split_path = os.path.join(DATA_DIR, "split_index.json")

if not os.path.exists(split_path):
    np.random.seed(SPLIT_SEED)
    all_idx = list(range(n_views))
    np.random.shuffle(all_idx)
    n_train = int(TRAIN_RATIO * n_views)
    train_idx = sorted(all_idx[:n_train])
    test_idx = sorted(all_idx[n_train:])

    with open(split_path, 'w') as f:
        json.dump({"train": train_idx, "test": test_idx}, f, indent=2)
    print(f"Split 생성: train={len(train_idx)}, test={len(test_idx)}")
else:
    with open(split_path) as f:
        split = json.load(f)
    print(f"기존 split 로드: train={len(split['train'])}, test={len(split['test'])}")

### 2.3 데이터 검증 & 시각화

In [ ]:
import json
import glob
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

# 파일 확인
checks = {
    "rgb images":    len(glob.glob(os.path.join(DATA_DIR, "rgb_*.png"))),
    "depth images":  len(glob.glob(os.path.join(DATA_DIR, "depth_raw_*.png"))),
    "calib files":   len(glob.glob(os.path.join(DATA_DIR, "calib_*.ini"))),
    "split_index.json": os.path.isfile(os.path.join(DATA_DIR, "split_index.json")),
}
for k, v in checks.items():
    print(f"  {k}: {v}")

with open(os.path.join(DATA_DIR, "split_index.json")) as f:
    split = json.load(f)
print(f"\nTrain: {len(split['train'])} views, Test: {len(split['test'])} views")

# 샘플 시각화 (RGB + Depth)
sample_ids = [0, 100, 300, 500, 640]
fig, axes = plt.subplots(2, len(sample_ids), figsize=(4 * len(sample_ids), 8))

for i, idx in enumerate(sample_ids):
    rgb = Image.open(os.path.join(DATA_DIR, f"rgb_{idx:04d}.png"))
    axes[0, i].imshow(rgb)
    label = "TRAIN" if idx in split["train"] else "TEST"
    axes[0, i].set_title(f"#{idx} ({label})", fontsize=10)
    axes[0, i].axis("off")

    depth_raw = cv2.imread(os.path.join(DATA_DIR, f"depth_raw_{idx:04d}.png"), cv2.IMREAD_UNCHANGED)
    depth_m = depth_raw.astype(np.float32) * 0.01
    depth_vis = np.where(depth_m > 0, depth_m, np.nan)
    axes[1, i].imshow(depth_vis, cmap="jet_r")
    axes[1, i].set_title(f"Depth [{depth_m[depth_m>0].min():.0f}-{depth_m[depth_m>0].max():.0f}m]", fontsize=9)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("RGB", fontsize=12, rotation=0, labelpad=40, va="center")
axes[1, 0].set_ylabel("Depth", fontsize=12, rotation=0, labelpad=40, va="center")
plt.suptitle(f"{SCENE_NAME} — 641 views, 800x800", fontsize=14)
plt.tight_layout()
plt.show()

---
## Section 3: Training

**Baseline 3DGS**: photometric loss만 (depth 없이)

**DRGS**: photometric + depth supervision (GT depth L1) + depth regularization (Canny smoothness)

### 3.1 사전 준비

In [ ]:
os.chdir(PROJECT_ROOT)
os.makedirs("debug", exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("준비 완료")

### 3.2 Baseline 3DGS 학습

In [ ]:
os.chdir(PROJECT_ROOT)

!python train.py \
    -s {DATA_DIR} \
    --eval \
    --port 6009 \
    --model_path {BASELINE_DIR} \
    --resolution {RESOLUTION} \
    --kshot {KSHOT} \
    --seed {SEED} \
    --iterations {ITERATIONS} \
    --test_iterations {TEST_ITERS} \
    --save_iterations {SAVE_ITERS}

### 3.3 DRGS 학습 (Depth Supervision + Depth Regularization)

`--depth`: GT depth와 렌더링 depth 간 L1 loss (가중치 0.5)

`--usedepthReg`: Canny edge 기반 depth smoothness L2 loss (가중치 1.0)

In [ ]:
os.chdir(PROJECT_ROOT)

!python train.py \
    -s {DATA_DIR} \
    --eval \
    --port 6010 \
    --model_path {DRGS_DIR} \
    --resolution {RESOLUTION} \
    --kshot {KSHOT} \
    --seed {SEED} \
    --iterations {ITERATIONS} \
    --test_iterations {TEST_ITERS} \
    --save_iterations {SAVE_ITERS} \
    --depth \
    --usedepthReg

### 3.4 TensorBoard (선택)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}

---
## Section 4: Rendering

In [ ]:
os.chdir(PROJECT_ROOT)

!python render.py -s {DATA_DIR} -m {BASELINE_DIR}
!python render.py -s {DATA_DIR} -m {DRGS_DIR}

print("렌더링 완료")

---
## Section 5: Evaluation

In [ ]:
os.chdir(PROJECT_ROOT)

!python metrics.py -m {BASELINE_DIR} {DRGS_DIR}

### 5.1 결과 비교 테이블

In [ ]:
import json
import pandas as pd

results = {}
for label, model_dir in [("Baseline 3DGS", BASELINE_DIR), ("DRGS", DRGS_DIR)]:
    rp = os.path.join(model_dir, "results.json")
    if os.path.exists(rp):
        with open(rp) as f:
            data = json.load(f)
        for method_name, metrics in data.items():
            results[label] = metrics
    else:
        print(f"[WARNING] {rp} not found")

if results:
    df = pd.DataFrame(results).T
    df = df[["PSNR", "SSIM", "LPIPS"]]
    df["PSNR"] = df["PSNR"].map("{:.2f}".format)
    df["SSIM"] = df["SSIM"].map("{:.4f}".format)
    df["LPIPS"] = df["LPIPS"].map("{:.4f}".format)
    print(f"\n=== {SCENE_NAME} | K={KSHOT} | Seed={SEED} ===\n")
    display(df)

### 5.2 학습 메트릭 추이

In [ ]:
import matplotlib.pyplot as plt

def parse_metric_txt(path):
    iters, psnrs, ssims, lpipss = [], [], [], []
    if not os.path.exists(path): return None
    with open(path) as f:
        for line in f:
            parts = line.strip().split("_")
            if len(parts) == 4:
                iters.append(int(parts[0]))
                psnrs.append(float(parts[1]))
                ssims.append(float(parts[2]))
                lpipss.append(float(parts[3]))
    return {"iter": iters, "psnr": psnrs, "ssim": ssims, "lpips": lpipss}

bm = parse_metric_txt(os.path.join(BASELINE_DIR, "metric.txt"))
dm = parse_metric_txt(os.path.join(DRGS_DIR, "metric.txt"))

if bm or dm:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, key, label in zip(axes, ["psnr","ssim","lpips"], ["PSNR (dB)","SSIM","LPIPS"]):
        if bm and bm[key]: ax.plot(bm["iter"], bm[key], "o-", label="Baseline", ms=4)
        if dm and dm[key]: ax.plot(dm["iter"], dm[key], "s-", label="DRGS", ms=4)
        ax.set_xlabel("Iteration"); ax.set_ylabel(label); ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle(f"{SCENE_NAME} | K={KSHOT}", fontsize=13)
    plt.tight_layout(); plt.show()

---
## Section 6: Visualization

### 6.1 GT / Baseline / DRGS 렌더링 비교

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import glob, os

def find_render_dir(model_dir):
    test_dir = os.path.join(model_dir, "test")
    if not os.path.isdir(test_dir): return None
    ours = sorted(glob.glob(os.path.join(test_dir, "ours_*")))
    return ours[-1] if ours else None

brd = find_render_dir(BASELINE_DIR)
drd = find_render_dir(DRGS_DIR)

if brd and drd:
    gts = sorted(glob.glob(os.path.join(brd, "gt", "*.png")))
    bfs = sorted(glob.glob(os.path.join(brd, "renders", "*.png")))
    dfs = sorted(glob.glob(os.path.join(drd, "renders", "*.png")))

    n = min(5, len(gts))
    fig, axes = plt.subplots(3, n, figsize=(4*n, 12))
    if n == 1: axes = axes.reshape(3, 1)

    for i in range(n):
        axes[0,i].imshow(Image.open(gts[i])); axes[0,i].set_title(f"GT #{i}"); axes[0,i].axis("off")
        axes[1,i].imshow(Image.open(bfs[i])); axes[1,i].set_title(f"Baseline #{i}"); axes[1,i].axis("off")
        axes[2,i].imshow(Image.open(dfs[i])); axes[2,i].set_title(f"DRGS #{i}"); axes[2,i].axis("off")

    for r, lbl in enumerate(["GT", "Baseline", "DRGS"]):
        axes[r,0].set_ylabel(lbl, fontsize=12, rotation=0, labelpad=60, va="center")
    plt.suptitle(f"Test View Comparison — {SCENE_NAME} K={KSHOT}", fontsize=14)
    plt.tight_layout(); plt.show()
else:
    print("렌더링 결과 없음. Section 4 실행 필요.")

### 6.2 Depth Map 비교

In [ ]:
import numpy as np
import matplotlib

def colorize_depth(d, cmap="jet_r"):
    valid = d > 0
    if valid.sum() == 0: return np.zeros((*d.shape, 3))
    norm = (d - d[valid].min()) / (d[valid].max() - d[valid].min() + 1e-8)
    norm[~valid] = 1.0
    c = matplotlib.cm.get_cmap(cmap)(norm)[:,:,:3]
    c[~valid] = 0.0
    return c

if brd and drd:
    bd_files = sorted(glob.glob(os.path.join(brd, "depth", "*.npy")))
    dd_files = sorted(glob.glob(os.path.join(drd, "depth", "*.npy")))

    if bd_files and dd_files:
        n = min(5, len(bd_files))
        fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
        if n == 1: axes = axes.reshape(2, 1)
        for i in range(n):
            axes[0,i].imshow(colorize_depth(np.load(bd_files[i]).squeeze()))
            axes[0,i].set_title(f"Baseline #{i}"); axes[0,i].axis("off")
            axes[1,i].imshow(colorize_depth(np.load(dd_files[i]).squeeze()))
            axes[1,i].set_title(f"DRGS #{i}"); axes[1,i].axis("off")
        axes[0,0].set_ylabel("Baseline\nDepth", fontsize=11, rotation=0, labelpad=70, va="center")
        axes[1,0].set_ylabel("DRGS\nDepth", fontsize=11, rotation=0, labelpad=70, va="center")
        plt.suptitle(f"Depth Comparison — {SCENE_NAME} K={KSHOT}", fontsize=14)
        plt.tight_layout(); plt.show()

---
## Section 7: 결과 저장

In [ ]:
import shutil

exp_name = f"{SCENE_NAME}_k{KSHOT}_s{SEED}"
save_dir = os.path.join(DRIVE_SAVE_DIR, exp_name)
os.makedirs(save_dir, exist_ok=True)

for label, src in [("baseline", BASELINE_DIR), ("drgs", DRGS_DIR)]:
    dst = os.path.join(save_dir, label)
    if os.path.exists(src):
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"[OK] {label} → {dst}")

print(f"\n저장 완료: {save_dir}")

In [ ]:
!echo "=== Disk ==="  && df -h / | tail -1
!echo "=== Output ===" && du -sh {OUTPUT_DIR}/* 2>/dev/null || echo "(empty)"
!echo "=== Drive ==="  && du -sh {DRIVE_SAVE_DIR}/* 2>/dev/null || echo "(empty)"